In [16]:
%%capture
pip install xgboost catboost

In [17]:
import warnings, math
warnings.filterwarnings("ignore")
import calendar
import datetime as dt

import numpy as np
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype, is_object_dtype, is_string_dtype
import matplotlib.pyplot as plt

from datetime import timedelta

# Métricas y visuales
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, precision_score, recall_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import TimeSeriesSplit,train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import confusion_matrix

# Opcionales (se activan si están instalados)
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostClassifier, Pool
    HAS_CAT = True
except Exception:
    HAS_CAT = False
    

    
ID_COLS = {
    "user_id", "userid",
    "channelUserIdentifier",
    "premia_accountid", "accountid", "member_id",
    "spin_user_id", "id"
}


In [18]:
from google.cloud import bigquery
client = bigquery.Client(project="spin-aip-singularity-comp-sb")

query = """
SELECT * 
FROM `spin-aip-singularity-comp-sb.model_activation.dataste_model_activation_long_timewindow_V-1-4-1` 
"""
data = client.query(query).to_dataframe()

In [19]:
TZ_MX = "America/Mexico_City"

# -------------------------
# Helpers de fechas/tiempo
# -------------------------
def to_dt_local(s: pd.Series | None, tz: str = TZ_MX) -> pd.Series | None:
    """Convierte a datetime TZ-aware en MX. Soporta UTC/naive/None."""
    if s is None:
        return None
    x = pd.to_datetime(s, errors="coerce")
    # Si ya tiene tz, convierte; si no, localiza en MX
    if getattr(x.dt, "tz", None) is not None:
        return x.dt.tz_convert(tz)
    # Caso BigQuery Date/naive: localiza a medianoche MX
    return x.dt.tz_localize(tz, nonexistent="NaT", ambiguous="NaT")

def month_end(ts: pd.Timestamp) -> pd.Timestamp:
    last = calendar.monthrange(ts.year, ts.month)[1]
    return pd.Timestamp(year=ts.year, month=ts.month, day=last)

def near_payday_flag(dates: pd.Series, window_days: int = 3) -> pd.Series:
    """
    Flag=1 si la fecha cae cerca (±window_days) de quincena (15) o fin de mes.
    """
    d = pd.to_datetime(dates, errors="coerce")
    fifteenth = d.apply(lambda x: pd.Timestamp(year=x.year, month=x.month, day=15))
    monthend  = d.apply(month_end)
    dist15 = (d - fifteenth).abs().dt.days
    distEOM = (d - monthend ).abs().dt.days
    return ((dist15 <= window_days) | (distEOM <= window_days)).astype(int)

def make_daypart(h: int) -> str:
    if h < 12: return "morning"
    if h < 18: return "afternoon"
    return "night"

# -------------------------
# Labels (discretas + acum)
# -------------------------
def build_time_window_labels(
    df: pd.DataFrame,
    instant_hours: int = 1,
    signup_date_col: str = "signup_date",
    first_tx_ts_col: str | None = None,       # ej. "first_tx_ts" si existe
    activation_date_col: str = "activation_date_ever"
) -> pd.DataFrame:
    """
    Crea y_w0, y_w1, y_w7, y_w30 y y_cum30 con la definición:
      - W0:  [0, instant_hours] horas
      - W1:  (instant_hours, 24]
      - W7:  (24, 24*7]
      - W30: (24*7, 24*30]
    y_cum30 = 1 si activó en [0..30d]
    """
    out = df.copy(deep=True)

    # Base temporal de signup (date -> tz-aware a medianoche MX)
    if signup_date_col not in out.columns:
        raise ValueError(f"Falta columna {signup_date_col} para labels.")
    signup_dt = pd.to_datetime(out[signup_date_col], errors="coerce")
    signup_ts = signup_dt.dt.tz_localize(TZ_MX)

    # Origen de activación: preferimos TS exacto si existe, si no, date
    if first_tx_ts_col and first_tx_ts_col in out.columns:
        first_tx_ts = to_dt_local(out[first_tx_ts_col])
        delta_h = (first_tx_ts - signup_ts).dt.total_seconds() / 3600.0
    elif activation_date_col in out.columns:
        act_dt  = pd.to_datetime(out[activation_date_col], errors="coerce")
        act_ts  = act_dt.dt.tz_localize(TZ_MX)
        delta_h = (act_ts - signup_ts).dt.total_seconds() / 3600.0
    elif "days_to_first_activation" in out.columns:
        # Fallback directo si ya tienes el delta en días
        delta_h = pd.to_numeric(out["days_to_first_activation"], errors="coerce") * 24.0
    else:
        raise ValueError("No encuentro TS/DATE de activación (first_tx_ts / activation_date_ever / days_to_first_activation).")

    y_w0  = ((delta_h >= 0) & (delta_h <= instant_hours)).astype(int)
    y_w1  = ((delta_h > instant_hours) & (delta_h <= 24)).astype(int)
    y_w7  = ((delta_h > 24) & (delta_h <= 24*7)).astype(int)
    y_w30 = ((delta_h > 24*7) & (delta_h <= 24*30)).astype(int)

    out["y_w0"]   = y_w0.fillna(0)
    out["y_w1"]   = y_w1.fillna(0)
    out["y_w7"]   = y_w7.fillna(0)
    out["y_w30"]  = y_w30.fillna(0)
    out["y_cum30"]= (out["y_w0"] + out["y_w1"] + out["y_w7"] + out["y_w30"]).clip(0,1)

    return out

# ---------------------------------
# Features censurados hasta k días
# ---------------------------------
def build_features_at_k(
    df: pd.DataFrame,
    k_days: int,
    event_windows: list[tuple[str, str, str]] | None = None
) -> pd.DataFrame:
    """
    Genera features solo con info <= t_k = signup_ts + k días.
    - KYC momentum: tiempo/flags a confirmación teléfono/email
    - Card readiness: link de tarjeta
    - Premia hook
    - Calendario: DOW, week, month, hour, daypart, near_payday
    - Demografía: edad/age_bucket
    - Baseline por estado (lag-1 mes) con y_cum30
    - Cruces: canal×daypart, canal×estado, edad×canal
    - Limpia variables con riesgo de fuga post-activación
    """
    out = df.copy(deep=True)

    # signup_ts/t_k base (local, aware)
    if "signup_ts" in out:
        out["signup_ts"] = to_dt_local(out["signup_ts"])
    else:
        # Si sólo hay signup_date, localiza a medianoche MX
        sd = pd.to_datetime(out["signup_date"], errors="coerce")
        out["signup_ts"] = sd.dt.tz_localize(TZ_MX)

    out["t_k"] = out["signup_ts"] + pd.to_timedelta(k_days, unit="D")

    # Normalización de fechas adicionales
    out["signup_date"] = pd.to_datetime(out.get("signup_date"), errors="coerce").dt.date
    if "birth_date" in out:
        out["birth_date"] = pd.to_datetime(out["birth_date"], errors="coerce").dt.date

    # Phone/Email confirm (censurados a k)
    if "phone_conf_ts" in out:
        out["phone_conf_ts"] = to_dt_local(out["phone_conf_ts"])
    if "email_conf_ts" in out:
        out["email_conf_ts"] = to_dt_local(out["email_conf_ts"])

    # Card linked (si viene como fecha)
    if "Card_linked_date" in out:
        card_dt = pd.to_datetime(out["Card_linked_date"], errors="coerce")
        out["card_linked_ts"] = card_dt.dt.tz_localize(TZ_MX)

    # --- KYC momentum (censurado a k)
    def censored_lag(ts_col):
        ts = out[ts_col]
        dt_h = (ts - out["signup_ts"]).dt.total_seconds() / 3600.0
        return np.where(ts.notna() & (ts <= out["t_k"]), np.maximum(dt_h, 0), np.nan)

    out["time_to_phone_confirm_k"] = np.nan
    out["time_to_email_confirm_k"] = np.nan
    out["has_phone_confirm_by_k"]  = 0
    out["has_email_confirm_by_k"]  = 0

    if "phone_conf_ts" in out:
        out["time_to_phone_confirm_k"] = censored_lag("phone_conf_ts")
        out["has_phone_confirm_by_k"]  = ((out["phone_conf_ts"].notna()) & (out["phone_conf_ts"] <= out["t_k"])).astype(int)
    if "email_conf_ts" in out:
        out["time_to_email_confirm_k"] = censored_lag("email_conf_ts")
        out["has_email_confirm_by_k"]  = ((out["email_conf_ts"].notna()) & (out["email_conf_ts"] <= out["t_k"])).astype(int)

    # --- Card readiness (censurado a k)
    out["card_linked_by_k"]   = 0
    out["time_to_card_link_k"]= np.nan
    if "card_linked_ts" in out:
        within_k = ((out["card_linked_ts"].notna()) & (out["card_linked_ts"] <= out["t_k"]))
        out["card_linked_by_k"]    = within_k.astype(int)
        dt_h = (out["card_linked_ts"] - out["signup_ts"]).dt.total_seconds() / 3600.0
        out["time_to_card_link_k"] = np.where(within_k, np.maximum(dt_h, 0), np.nan)

    # --- Premia hook
    out["premia_present_at_signup"] = (
        (out["premia_accountid"].notna() if "premia_accountid" in out else False) |
        ((out["has_premia"] == 1)      if "has_premia"        in out else False)
    ).astype(int)

    # --- Calendario / operación
    out["signup_local"] = out["signup_ts"]  # alias semántico
    out["signup_dow"]   = out["signup_local"].dt.dayofweek.astype("Int64")
    out["signup_week"]  = out["signup_local"].dt.isocalendar().week.astype(int)
    out["signup_month"] = out["signup_local"].dt.month.astype(int)
    out["signup_hour"]  = out["signup_local"].dt.hour.astype(int)
    out["daypart"]      = out["signup_hour"].apply(make_daypart)
    out["near_payday_k"]= near_payday_flag(pd.to_datetime(out["signup_local"].dt.date))

    # Eventos exógenos (opcionales)
    if event_windows:
        for name, start, end in event_windows:
            mask = (
                (out["signup_local"].dt.date >= pd.to_datetime(start).date()) &
                (out["signup_local"].dt.date <= pd.to_datetime(end ).date())
            )
            out[f"event_{name}"] = mask.astype(int)

    # --- Demografía / Geo
    if "birth_date" in out:
        age_days = (pd.to_datetime(out["signup_local"].dt.date) - pd.to_datetime(out["birth_date"]))
        out["age_at_signup"] = np.floor(age_days.dt.days / 365.25).astype("float")
        bins   = [18,25,35,45,200]
        labels = ["18-24","25-34","35-44","45+"]
        out["age_bucket"] = pd.cut(out["age_at_signup"], bins=bins, labels=labels, right=False)

    # --- Baseline lag por estado (usa y_cum30 del MES PREVIO)
    if {"stateName", "y_cum30"}.issubset(out.columns):
        tmp = out[["stateName","signup_local","y_cum30"]].copy()
        tmp["signup_month_key"] = pd.to_datetime(tmp["signup_local"].dt.to_period("M").astype(str))
        rates = (
            tmp.groupby(["stateName","signup_month_key"])["y_cum30"]
               .mean().reset_index().rename(columns={"y_cum30":"state_rate_curr"})
        )
        lag = rates.copy()
        # Desplaza la llave al MES SIGUIENTE para que al unir quede como lag-1
        lag["signup_month_key"] = lag["signup_month_key"] + pd.offsets.MonthBegin(1)
        lag = lag.rename(columns={"state_rate_curr":"state_rate_lag1"})

        out["signup_month_key"] = pd.to_datetime(out["signup_local"].dt.to_period("M").astype(str))
        out = out.merge(lag, on=["stateName","signup_month_key"], how="left")

        # Fallback global (mes previo)
        global_rates = (
            tmp.groupby("signup_month_key")["y_cum30"]
               .mean().reset_index().rename(columns={"y_cum30":"global_rate"})
        )
        global_rates["signup_month_key"] = global_rates["signup_month_key"] + pd.offsets.MonthBegin(1)
        out = out.merge(global_rates, on="signup_month_key", how="left")

        out["state_activation_baseline_lag1"] = (
            out["state_rate_lag1"].fillna(out["global_rate"]).fillna(0.0)
        )

    # --- Cruces pro-señal (si existen insumos)
    if "channelDetail" in out and "daypart" in out:
        out["x_channel_daypart"] = out["channelDetail"].astype(str) + "__" + out["daypart"].astype(str)
    if "channelDetail" in out and "stateName" in out:
        out["x_channel_state"] = out["channelDetail"].astype(str) + "__" + out["stateName"].astype(str)
    if "age_bucket" in out and "channelDetail" in out:
        out["x_age_channel"] = out["age_bucket"].astype(str) + "__" + out["channelDetail"].astype(str)

    # --- Drop: columnas con alto riesgo de fuga
    drop_cols = [
        "activation_date_ever","activation_date_30d","label_activated_30d",
        "tx_30d_count","tx_30d_amount","label_5tx_30d","first_tx_type",
        "first_tx_amount","activation_channel","latest_tx_date",
        "lifespan_days","days_since_last","tx_30d_from_activation",
        "days_to_first_activation",
        # auxiliares temporales que no quieres en X
        "card_linked_ts","state_rate_lag1","state_rate_curr","global_rate"
    ]
    out = out.drop(columns=[c for c in drop_cols if c in out.columns], errors="ignore")

    return out

# ---------------------------------------
# Builder maestro: k in {0,1,7} (producción)
# ---------------------------------------
def build_all_features(
    df: pd.DataFrame,
    ks=(0,1,7),
    event_windows: list[tuple[str, str, str]] | None = None
) -> dict[int, pd.DataFrame]:
    """
    Devuelve {k: df_features_at_k} con anti-leak aplicado.
    """
    out = {}
    for k in ks:
        out[k] = build_features_at_k(df, k_days=k, event_windows=event_windows)
    return out

# ========================
#  EXEC: lab + features
# ========================
# 1) Deriva labels (si no traes y_*). Ajusta los nombres si tu tabla usa otros.
df_lab = build_time_window_labels(
    data,
    instant_hours=1,
    signup_date_col="signup_date",
    first_tx_ts_col=None,                 # e.g. "first_tx_ts" si existiera
    activation_date_col="activation_date_ever"
)

# 2) Genera features para k=0,1,7
features_dict = build_all_features(df_lab, ks=(0,1,7))
feat_k0 = features_dict[0].copy()
feat_k1 = features_dict[1].copy()
feat_k7 = features_dict[7].copy()

# 3) Columnas candidatas a X (ejemplo para k=0)
LEAKY_COLS = {"t_k","signup_local","signup_month_key","signup_ts"}  # auxiliares fuera de X
LABELISH   = {"y_w0","y_w1","y_w7","y_w30","y_cum30"}               # no entrenables
X_cols = [c for c in feat_k0.columns if c not in LEAKY_COLS | LABELISH]

print("feat_k0 shape:", feat_k0.shape)
print("feat_k1 shape:", feat_k1.shape)
print("feat_k7 shape:", feat_k7.shape)
print("X_cols (sample):", X_cols[:15])


feat_k0 shape: (5740312, 50)
feat_k1 shape: (5740312, 50)
feat_k7 shape: (5740312, 50)
X_cols (sample): ['user_id', 'signup_date', 'userTypeIdentifier', 'channelUserIdentifier', 'accountLevel', 'stateName', 'gender', 'user_type', 'channelDetail', 'birth_date', 'birthState', 'Card_linked_date', 'IsActive', 'phn_confir', 'email_confir']


In [20]:
feat_k0 = feat_k0.loc[:, ~feat_k0.columns.duplicated()].copy()
feat_k1 = feat_k1.loc[:, ~feat_k1.columns.duplicated()].copy()
feat_k7 = feat_k7.loc[:, ~feat_k7.columns.duplicated()].copy()


In [21]:
# ========= RE-DECLARE: PurgedTimeSeriesCV (robusto) =========
TIME_COL = "signup_local"  # ya lo traes del factory

class PurgedTimeSeriesCV:
    """
    Forward-chaining CV por tiempo, con gap opcional antes del fold de validación
    y embargo opcional después. Entrena SOLO con pasado (sin futuro).
    """
    def __init__(self, n_splits=5, time_col=TIME_COL, gap_before_days=0, embargo_after_days=0):
        self.n_splits = int(n_splits)
        self.time_col = str(time_col)
        self.gap_before_days = int(gap_before_days)
        self.embargo_after_days = int(embargo_after_days)

    def split(self, X: pd.DataFrame, y=None):
        # 1) Garantiza una sola columna temporal
        if self.time_col not in X.columns:
            raise KeyError(f"TIME_COL '{self.time_col}' no está en X.columns.")
        x_time = X.loc[:, [self.time_col]]
        if x_time.shape[1] > 1:  # si hubo duplicados de nombre, quédate con la primera
            x_time = x_time.iloc[:, [0]]

        # 2) A datetime seguro (soporta db_dtypes.*)
        s_time = pd.to_datetime(x_time.iloc[:, 0], errors="coerce")

        # 3) Timestamps únicos y bounds por cuantiles
        t_sorted = s_time.sort_values()
        uniq_times = pd.Series(t_sorted).dropna().unique()
        if len(uniq_times) < self.n_splits + 1:
            raise ValueError("Muy pocos timestamps únicos para el n_splits solicitado.")

        # Índices de cortes (n_splits+1 “bordes”)
        cut_idx = np.linspace(0, len(uniq_times) - 1, self.n_splits + 1, dtype=int)

        for i in range(self.n_splits):
            val_start = uniq_times[cut_idx[i]]
            val_end   = uniq_times[cut_idx[i+1]] if i < self.n_splits - 1 else uniq_times[-1]

            # Validación: [val_start, val_end) salvo el último que es [val_start, +inf)
            if i < self.n_splits - 1:
                val_mask = (s_time >= val_start) & (s_time < val_end)
            else:
                val_mask = (s_time >= val_start)

            # Gap antes del inicio de validación
            gap_start = val_start - pd.Timedelta(days=self.gap_before_days)

            # Entrenamiento: estrictamente pasado (y respetando el gap)
            train_mask = (s_time < gap_start)

            # Embargo opcional post-validación (no es necesario si ya usamos solo pasado,
            # lo dejamos para futuras variantes si miras ventanas amplias)
            # embargo_end = val_end + pd.Timedelta(days=self.embargo_after_days)

            tr_idx = np.where(train_mask)[0]
            va_idx = np.where(val_mask)[0]

            # Sanity: evita folds vacíos
            if len(tr_idx) == 0 or len(va_idx) == 0:
                continue
            yield tr_idx, va_idx

# ====== EXTRA: asegúrate de no tener columnas duplicadas en X de cada horizonte
def _dedup_cols(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ~df.columns.duplicated()].copy()

In [22]:
# === 1) Métricas/visuales/umbral/fairness ===
def auprc(y_true, y_proba):
    return average_precision_score(y_true, y_proba)

def gains_curve(y_true, y_proba, target_class=0):
    scores = y_proba if target_class==1 else (1 - y_proba)
    order  = np.argsort(scores)[::-1]
    y      = (y_true == target_class).astype(int)[order]
    cum_evt = np.cumsum(y)
    perc_pop = np.arange(1, len(y)+1)/len(y)
    gains = cum_evt / max(1, y.sum())
    lift  = gains / np.maximum(perc_pop, 1e-12)
    return perc_pop, gains, lift

def reliability_plot(y_true, y_proba, n_bins=10, ax=None, title="Reliability"):
    prob_true, prob_pred = calibration_curve(y_true, y_proba, n_bins=n_bins, strategy="quantile")
    if ax is None:
        plt.figure(); ax = plt.gca()
    ax.plot(prob_pred, prob_true, marker="o", label="Empírico")
    ax.plot([0,1],[0,1],"--", label="Perfecto")
    ax.set_xlabel("Score promedio por bin"); ax.set_ylabel("Tasa real por bin")
    ax.set_title(title); ax.legend() 
    

def expected_cost(y_true, y_score, thr, cost_fp=1.0, cost_fn=5.0):
    """Costo esperado = FP*cost_fp + FN*cost_fn."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_score) >= thr).astype(int)
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return fp * cost_fp + fn * cost_fn


def best_threshold_by_cost(y_true, y_proba,
                           cost_fp=1.0, cost_fn=5.0,
                           qgrid=None,
                           min_q=0.05, max_q=0.95):
    """
    Versión saneada: busca el umbral que minimiza costo
    SOLO en el rango central de las probabilidades.
    """
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)

    if qgrid is None:
        qgrid = np.linspace(min_q, max_q, 19)

    cand = np.unique(np.quantile(y_proba, qgrid))
    best_thr = 0.5
    best_cost = float("inf")
    best_stats = (0, 0, 0, 0)

    for t in cand:
        y_hat = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
        cost = cost_fp*fp + cost_fn*fn
        if cost < best_cost:
            best_cost = cost
            best_thr = float(t)
            best_stats = (tn, fp, fn, tp)

    tn, fp, fn, tp = best_stats
    print(f"[COST] Best threshold={best_thr:.3f} | cost={best_cost:.2f} | tn={tn}, fp={fp}, fn={fn}, tp={tp}")
    return best_thr


def thresholds_by_group(y_true, y_proba, groups,
                        cost_fp=1.0, cost_fn=5.0,
                        min_support=3000):
    """
    Calcula umbral global y, si hay datos suficientes,
    umbral por grupo (canal).
    """
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)
    groups = np.asarray(groups).astype(str)

    thr_global = best_threshold_by_cost(
        y_true, y_proba,
        cost_fp=cost_fp, cost_fn=cost_fn,
        min_q=0.05, max_q=0.95
    )
    thr_map = {"_GLOBAL_": float(thr_global)}

    # por grupo
    for g in np.unique(groups):
        idx = (groups == g)
        if idx.sum() < min_support:
            continue
        thr_g = best_threshold_by_cost(
            y_true[idx], y_proba[idx],
            cost_fp=cost_fp, cost_fn=cost_fn,
            min_q=0.05, max_q=0.95
        )
        thr_map[g] = float(thr_g)

    return thr_map


def eval_at_thr_grouped(y_true, y_proba, groups, thr_map):
    """
    Mide P/R/NPV usando el umbral de cada grupo.
    Si el grupo no está en el mapa, usa el global.
    """
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)
    if groups is None:
        groups = np.array(["_GLOBAL_"] * len(y_true))
    else:
        groups = np.asarray(groups).astype(str)

    thr_global = thr_map.get("_GLOBAL_", 0.5)
    y_hat = np.zeros(len(y_true), dtype=int)
    for g in np.unique(groups):
        idx = (groups == g)
        thr_g = thr_map.get(g, thr_global)
        y_hat[idx] = (y_proba[idx] >= thr_g).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    prec = tp / (tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp / (tp+fn) if (tp+fn)>0 else 0.0
    npv  = tn / (tn+fn) if (tn+fn)>0 else 0.0

    return {
        "precision": prec,
        "recall": rec,
        "npv": npv,
        "tn": int(tn), "fp": int(fp),
        "fn": int(fn), "tp": int(tp)
    }
   
    
#def best_threshold_by_cost(y_true, y_proba, cost_fp=1.0, cost_fn=5.0):
  #  thrs = np.linspace(0,1,1001)
 #   best = (0.5, float("inf"), 0,0,0,0)
#    for t in thrs:
      #  y_hat = (y_proba >= t).astype(int)
     #   tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    #    cost = cost_fp*fp + cost_fn*fn
   #     if cost < best[1]:
  #          best = (t, cost, tn, fp, fn, tp)
 #   print(f"[COST] Best threshold={best[0]:.3f} | cost={best[1]:.2f} | tn={best[2]}, fp={best[3]}, fn={best[4]}, tp={best[5]}")
#    return best[0]


def best_threshold_max_npv(y_true, y_proba, max_points=200):
    """
    Calcula el umbral que maximiza el NPV pero en O(N log N),
    no en O(N × #umbrales).

    y_true: array-like binario (0/1)
    y_proba: probabilidades del modelo
    max_points: cuántos puntos de corte evaluar como máximo
    """
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)

    n = y_true.shape[0]

    # 1) ordenamos de mayor proba a menor
    order = np.argsort(-y_proba)
    y_true_sorted = y_true[order]
    proba_sorted  = y_proba[order]

    # 2) acumulados hacia adelante (para la parte "positiva")
    cum_pos = np.cumsum(y_true_sorted)          # TP si corto aquí
    cum_all = np.arange(1, n+1)                 # predichos positivos si corto aquí

    # 3) acumulados hacia atrás (para la parte "negativa")
    #   suffix_pos[k] = cuántos 1's hay de k hasta el final
    suffix_pos = np.cumsum(y_true_sorted[::-1])[::-1]

    # 4) si corto en k → predichos positivos = k, predichos negativos = n-k
    #   FN = positivos en la cola = suffix_pos[k]
    #   TN = (n-k) - FN
    ks = np.linspace(0, n-1, num=min(max_points, n), dtype=int)

    best_npv = -1.0
    best_thr = 0.5  # default por si todo sale 0

    for k in ks:
        neg_count = n - k
        if neg_count == 0:
            continue  # no hay negativos predichos → NPV no aplica

        fn = suffix_pos[k]
        tn = neg_count - fn
        denom = tn + fn  # == neg_count
        npv = tn / denom if denom > 0 else 0.0

        if npv > best_npv:
            best_npv = npv
            # el umbral es la proba del último que metimos como positivo
            best_thr = proba_sorted[k]

    return best_thr, best_npv


def fairness_report(y_true, y_proba, groups: pd.Series, thr: float):
    y_hat = (y_proba >= thr).astype(int)
    rep = []
    groups_series = pd.Series(groups).astype("category")
    for g in groups_series.cat.categories:
        idx = np.where(groups_series == g)[0]
        if len(idx) < 50:  # evita ruido
            continue
        yt = y_true[idx]; yh = y_hat[idx]
        tn, fp, fn, tp = confusion_matrix(yt, yh, labels=[0,1]).ravel()
        tpr = tp/max(1, (tp+fn))
        fpr = fp/max(1, (fp+tn))
        rep.append({"group": g, "n": len(idx), "TPR": tpr, "FPR": fpr})
    return pd.DataFrame(rep).sort_values("n", ascending=False)


In [23]:
def fit_group_isotonic(df_train, score_col, y_col, group_col, min_support=2000):
    models = {}
    # Fallback global
    models["_GLOBAL_"] = IsotonicRegression(out_of_bounds="clip").fit(
        df_train[score_col].values, df_train[y_col].values
    )
    for g, d in df_train.groupby(group_col):
        if len(d) < min_support:
            continue
        ir = IsotonicRegression(out_of_bounds="clip").fit(
            d[score_col].values, d[y_col].values
        )
        models[g] = ir
    return models

def predict_group_isotonic(df, score_col, group_col, models):
    global_m = models["_GLOBAL_"]
    out = np.empty(len(df), dtype=float)
    for g, idx in df.groupby(group_col).groups.items():
        m = models.get(g, global_m)
        out[idx] = m.predict(df.loc[idx, score_col].values)
    return out


In [24]:
# Ejemplo: ajusta a tu realidad
COST_MAP = {
    "POS": (1, 2),            # (cost_fp, cost_fn) push barato
    "DIGITAL_ORGANIC": (2, 2),
    "SPIN_PREMIA": (6, 1),    # si es caro contactar / incentivar
    "ORGANIC": (2, 2),
}
GLOBAL_COST_FP, GLOBAL_COST_FN = 2, 2


In [25]:
def thresholds_by_group(df_val, score_col, y_col, group_col,
                        cost_map=None, global_cost_fp=1.0, global_cost_fn=5.0,
                        min_support=3000):
    """Devuelve dict de umbrales por grupo + '_GLOBAL_'."""
    y = df_val[y_col].values
    s = df_val[score_col].values
    # Umbral global como fallback
    t_global = best_threshold_by_cost(
        y, s, cost_fp=global_cost_fp, cost_fn=global_cost_fn,
        qgrid=np.linspace(0.05, 0.95, 19),
        min_thr=np.quantile(s, 0.05)
    )
    thr_map = {"_GLOBAL_": float(t_global)}

    for g, d in df_val.groupby(group_col):
        if len(d) < min_support:   # evita sobreajuste en grupos chicos
            continue
        cfp, cfn = (cost_map.get(g, (global_cost_fp, global_cost_fn))
                    if cost_map else (global_cost_fp, global_cost_fn))
        t_g = best_threshold_by_cost(
            d[y_col].values, d[score_col].values,
            cost_fp=cfp, cost_fn=cfn,
            qgrid=np.linspace(0.05, 0.95, 19),
            min_thr=np.quantile(d[score_col].values, 0.05)
        )
        thr_map[g] = float(t_g)
    return thr_map

def apply_group_threshold(df, score_col, group_col, thr_map):
    t_global = thr_map.get("_GLOBAL_", 0.5)
    thr_vec = df[group_col].map(thr_map).fillna(t_global).values
    return (df[score_col].values >= thr_vec).astype(int)


In [26]:
# === 2) Preprocesamiento para modelos sklearn (OHE) ===
from sklearn.utils.validation import check_is_fitted as _tmp

def split_num_cat(df: pd.DataFrame, feats):
    num_cols = [c for c in feats if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])]
    cat_cols = [c for c in feats if (pd.api.types.is_categorical_dtype(df[c]) or df[c].dtype == object)]
    return num_cols, cat_cols

def onehot_preprocessor(num_cols, cat_cols, scale_numeric=False, min_freq=0.01, dense_ohe=False):
    num_pipe = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_pipe.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(num_pipe)

    # Compatibilidad scikit: sparse_output (>=1.2) / sparse (antiguas)
    if dense_ohe:
        try:
            ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=min_freq, sparse_output=False)
        except TypeError:
            ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=min_freq, sparse=False)
    else:
        ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=min_freq)  # por defecto (sparse)

    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", ohe)])

    pre = ColumnTransformer(
        transformers=[("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)],
        remainder="drop",
        sparse_threshold=0.3  # no importa si OHE ya es denso
    )
    return pre


In [27]:

# === 3) Builders de modelos ===
def make_logreg(pre):
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])

def make_rf(pre):
    return Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=600, max_depth=None, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=42))])

def make_histgb(pre):
    # Convierte a denso si viene sparse (solo para HistGB)
    to_dense = FunctionTransformer(lambda X: X.toarray() if hasattr(X, "toarray") else X,
                                   accept_sparse=True)
    return Pipeline([
        ("pre", pre),
        ("to_dense", to_dense),
        ("clf", HistGradientBoostingClassifier(
            max_depth=None,
            max_leaf_nodes=127,
            max_bins = 127,
            min_samples_leaf= 100,
            learning_rate= 0.05,
            l2_regularization=0.0,
            early_stopping=True,
            random_state=42
        ))
    ])



def make_xgb_native():
    if not HAS_XGB: return None
    return xgb.XGBClassifier(
        n_estimators=600, max_depth=6, learning_rate=0.07,
        subsample=0.9, colsample_bytree=0.9,
        tree_method="hist", enable_categorical=True,
        reg_lambda=1.0, reg_alpha=0.0, random_state=42, n_jobs=-1
    )

def make_lgbm_native():
    if not HAS_LGBM: return None
    return lgb.LGBMClassifier(
        n_estimators=1200, learning_rate=0.05, num_leaves=63,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=42, n_jobs=-1
    )

def make_cat_native():
    if not HAS_CAT: return None
    return CatBoostClassifier(
        depth=8, learning_rate=0.07, iterations=1200,
        loss_function="Logloss", eval_metric="AUC",
        l2_leaf_reg=3.0, random_seed=42,
        auto_class_weights="Balanced", verbose=False
    )


In [28]:
# === 4) Selector de X,y por horizonte (anti-fuga por k) ===
def get_xy_for_horizon(hname: str, label_col: str):
    # Asegura y_cum7 si no existe
    if "y_cum7" not in df_lab.columns and set(["y_w0","y_w1","y_w7"]).issubset(df_lab.columns):
        df_lab["y_cum7"] = ((df_lab["y_w0"] + df_lab["y_w1"] + df_lab["y_w7"])>0).astype(int)

    # Mapeo horizonte -> features k
    k_map = {
        "W0_discrete": 0,
        "W1_discrete": 0,   # as-of fin de D0
        "W7_discrete": 1,   # as-of fin de D1
        "W30_discrete": 7,  # as-of fin de D7
        "W7_cum": 0,
        "W30_cum": 0
    }
    k = k_map[hname]
    feat = {0: feat_k0, 1: feat_k1, 7: feat_k7}[k].copy()

    # Labels vienen embebidas en feat_* (por herencia de df_lab). Si no, merge por índice.
    if label_col not in feat.columns and label_col in df_lab.columns:
        feat[label_col] = df_lab[label_col]

    # Drop auxiliares y labels de X
    LEAKY_COLS = {"t_k","signup_ts","signup_local","signup_month_key"}
    LABELISH   = {"y_w0","y_w1","y_w7","y_w30","y_cum7","y_cum30"}
    
    feats = [c for c in feat.columns if c not in (LEAKY_COLS | LABELISH | ID_COLS)]
    feats = [c for c in feats if feat[c].notna().sum() > 0] 

 

    # Num/Cat
    num_cols, cat_cols = split_num_cat(feat, feats)

    # Forzar category en object para nativos
    for c in cat_cols:
        if feat[c].dtype == object:
            feat[c] = feat[c].astype("category")

    # Construye X e impide duplicados
    X = feat[feats + [TIME_COL]].copy()
    # Si por accidente feats traía TIME_COL, aquí nos aseguramos de dejarlo 1 sola vez
    X = X.loc[:, ~X.columns.duplicated()].copy()

    # Sanity check: que TIME_COL sea único y exista
    time_hits = (X.columns == TIME_COL).sum()
    assert time_hits == 1, f"TIME_COL '{TIME_COL}' aparece {time_hits} veces en X. Revisa duplicados."
    y = feat[label_col].astype(int).values
    return X, y, feats, num_cols, cat_cols, k

In [29]:
# === 5) Entrenamiento/Evaluación por horizonte ===
def train_eval_horizon(hname: str, label_col: str,
                       gap_before_days: int = 1, embargo_after_days: int = 30,
                       cost_fp: float = 5.0, cost_fn: float = 1.0,
                       calib_method: str = "isotonic"):
    print(f"\n=== Horizonte: {hname} | label={label_col} ===")
    X, y, feats, num_cols, cat_cols, k = get_xy_for_horizon(hname, label_col)

    # Preprocesador para modelos sklearn
    pre = onehot_preprocessor(num_cols, cat_cols, scale_numeric=False, min_freq=0.01)
    pre_sparse = onehot_preprocessor(num_cols, cat_cols, scale_numeric=False, min_freq=0.01, dense_ohe=False)
    pre_dense  = onehot_preprocessor(num_cols, cat_cols, scale_numeric=False, min_freq=0.01, dense_ohe=True)


    # Model zoo
    models = {
#        "LogReg": make_logreg(pre),
#        "RandomForest": make_rf(pre),
        "HistGB": make_histgb(pre_dense)
    }
#    if HAS_XGB: models["XGBoost"] = make_xgb_native()
#    if HAS_LGBM: models["LightGBM"] = make_lgbm_native()
#    if HAS_CAT: models["CatBoost"] = make_cat_native()

    # CV temporal (gap/embargo configurable)
    cv = PurgedTimeSeriesCV(n_splits=5, time_col=TIME_COL,
                            gap_before_days=gap_before_days,
                            embargo_after_days=embargo_after_days)


    results = {}
    for mname, model in models.items():
        if model is None:
            print(f"[{mname}] no disponible.")
            continue

        print(f"\n-- Modelo: {mname} --")
        oof_proba = np.zeros(len(X)); oof_mask = np.zeros(len(X), dtype=bool)
        use_native_cat = mname in ["XGBoost","LightGBM","CatBoost"]

        for fold, (tr_idx, va_idx) in enumerate(cv.split(X)):
            Xtr, Xva = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
            ytr, yva = y[tr_idx], y[va_idx]

            if use_native_cat:
                # balanceo simple
                pos = max(1, ytr.sum()); neg = max(1, len(ytr)-pos)
                if mname == "XGBoost":
                    model.set_params(scale_pos_weight=neg/pos)
                    model.fit(Xtr[feats], ytr, eval_set=[(Xva[feats], yva)], verbose=False)
                    proba = model.predict_proba(Xva[feats])[:,1]
                elif mname == "LightGBM":
                    model.set_params(class_weight={0:1.0, 1:neg/pos})
                    model.fit(Xtr[feats], ytr,
                              eval_set=[(Xva[feats], yva)],
                              categorical_feature=[c for c in cat_cols if c in feats],
                              callbacks=[lgb.early_stopping(50, verbose=False)])
                    proba = model.predict_proba(Xva[feats])[:,1]
                elif mname == "CatBoost":
                    train_pool = Pool(Xtr[feats], label=ytr, cat_features=[c for c in cat_cols if c in feats])
                    valid_pool = Pool(Xva[feats], label=yva, cat_features=[c for c in cat_cols if c in feats])
                    model.fit(train_pool, eval_set=valid_pool, verbose=False)
                    proba = model.predict_proba(valid_pool)[:,1]
            else:
                model.fit(Xtr[feats], ytr)
                proba = model.predict_proba(Xva[feats])[:,1]

            oof_proba[va_idx] = proba
            oof_mask[va_idx] = True

            ap = auprc(yva, proba); auc = roc_auc_score(yva, proba)
            print(f"   Fold {fold+1}: AUC-PR={ap:.4f} | AUC-ROC={auc:.4f}")

        y_true = y[oof_mask]; y_prob = oof_proba[oof_mask]
        ap = auprc(y_true, y_prob); auc = roc_auc_score(y_true, y_prob)
        print(f"[{mname}] OOF: AUC-PR={ap:.4f} | AUC-ROC={auc:.4f}")

        # Calibración rápida (global)
        try:
            if use_native_cat:
                cal = CalibratedClassifierCV(model, cv=3, method=calib_method)
                cal.fit(X[feats], y)
                y_prob_cal = cal.predict_proba(X[feats])[:,1]
                brier = brier_score_loss(y, y_prob_cal)
                print(f"[{mname}] Brier (calibrado): {brier:.4f}")
            else:
                cal = CalibratedClassifierCV(model, cv=3, method=calib_method)
                cal.fit(X[feats], y)
                y_prob_cal = cal.predict_proba(X[feats])[:,1]
                brier = brier_score_loss(y, y_prob_cal)
                print(f"[{mname}] Brier (calibrado): {brier:.4f}")
        except Exception as e:
            print(f"[{mname}] Calibración saltada ({e}).")
            y_prob_cal = y_prob
        #
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(y_prob, y_true)
        y_prob_cal = iso.predict(y_prob)
        brier = brier_score_loss(y_true, y_prob_cal)
        print(f"[{mname}] Brier (isotonic OOF): {brier:.4f}")

        # a partir de aquí TODO usa y_prob_cal, no y_prob
        thr_cost = best_threshold_by_cost(
            y_true, y_prob_cal,
            cost_fp=cost_fp, cost_fn=cost_fn
        )
        thr_npv, npv_val = best_threshold_max_npv(y_true, y_prob, max_points=200)
        res = {
        "horizon": hname,
        "label_col": label_col,
        "gap_before_days": gap_before_days,
        "embargo_after_days": embargo_after_days,
        # puedes tener aquí tus métricas base (auc, pr, etc.) si ya las calculas arriba
    }
            # 4) guardas los thresholds
        res["thr_cost"] = float(thr_cost)
        res["thr_npv"] = float(thr_npv)
        res["npv_at_thr_npv"] = float(npv_val)
        

        # Umbrales
        thr_cost = best_threshold_by_cost(y_true, y_prob, cost_fp=cost_fp, cost_fn=cost_fn)
        thr_npv, npv_val = best_threshold_max_npv(y_true, y_prob, max_points=200)
        return res


        
        def eval_at_thr(y_true, y_prob, thr):
            y_hat = (y_prob >= thr).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
            prec = precision_score(y_true, y_hat, zero_division=0)
            rec  = recall_score(y_true, y_hat, zero_division=0)
            npv  = tn/(tn+fn) if (tn+fn)>0 else 0.0
            return {
                "precision":prec, "recall":rec, "npv":npv,
                "tn":tn, "fp":fp, "fn":fn, "tp":tp
            }

        eval_cost = eval_at_thr(y_true, y_prob_cal, thr_cost)
        eval_npv  = eval_at_thr(y_true, y_prob_cal, thr_npv)

        print(f"[{mname}] @thr_cost -> P={eval_cost['precision']:.3f} | R={eval_cost['recall']:.3f} | NPV={eval_cost['npv']:.3f}")
        print(f"[{mname}] @thr_npv  -> P={eval_npv['precision']:.3f} | R={eval_npv['recall']:.3f} | NPV={eval_npv['npv']:.3f}")

        # Visuales clave
        fig, axs = plt.subplots(2, 2, figsize=(12,10))

        # PR
        precs, recalls, _ = precision_recall_curve(y_true, y_prob)
        axs[0,0].plot(recalls, precs); axs[0,0].set_title(f"{mname} - PR (AP={ap:.3f})")
        axs[0,0].set_xlabel("Recall"); axs[0,0].set_ylabel("Precision")

        # ROC
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        axs[0,1].plot(fpr, tpr); axs[0,1].plot([0,1],[0,1],'--')
        axs[0,1].set_title(f"{mname} - ROC (AUC={auc:.3f})")
        axs[0,1].set_xlabel("FPR"); axs[0,1].set_ylabel("TPR")

        # Confusión @thr_cost
        y_hat_cost = (y_prob >= thr_cost).astype(int)
        cm = confusion_matrix(y_true, y_hat_cost, normalize='true')
        im = axs[1,0].imshow(cm, cmap="Blues")
        axs[1,0].set_title(f"{mname} - Confusion @thr_cost")
        axs[1,0].set_xticks([0,1]); axs[1,0].set_xticklabels(["0","1"])
        axs[1,0].set_yticks([0,1]); axs[1,0].set_yticklabels(["0","1"])
        for (i,j), v in np.ndenumerate(cm):
            axs[1,0].text(j, i, f"{v:.2f}", ha="center", va="center")

        # Reliability
        reliability_plot(y_true, y_prob, n_bins=10, ax=axs[1,1], title=f"{mname} - Reliability")
        plt.tight_layout(); plt.show()

        # Gains / Lift para clase 0 (Nudge)
        ppop, gains, lift = gains_curve(y_true, y_prob, target_class=0)
        plt.figure(figsize=(6,4)); plt.plot(ppop, gains); plt.title(f"{mname} - Gains (clase 0)")
        plt.xlabel("% población"); plt.ylabel("% capturado clase 0"); plt.show()
        plt.figure(figsize=(6,4)); plt.plot(ppop, lift); plt.title(f"{mname} - Lift (clase 0)")
        plt.xlabel("% población"); plt.ylabel("Lift"); plt.show()

        # Fairness rápido por columnas sensibles (si existen en X original)
        SENSITIVE_COLS = [c for c in ["gender","stateName","channelDetail"] if c in X.columns]
        fairness_tables = {}
        for gcol in SENSITIVE_COLS:
            df_fair = fairness_report(
                y_true, y_prob_cal,
                groups=X.loc[X.index[oof_mask], gcol],
                thr=thr_cost
            )
            if not df_fair.empty:
                print(f"\n[{mname}] Fairness @thr_cost por {gcol} (top grupos):")
                display(df_fair.head(10))
            fairness_tables[gcol] = df_fair

        results[mname] = {
            "model": model,
            "oof_ap": ap, "oof_auc": auc,
            "thr_cost": thr_cost,
#            "thr_cost_map": thr_cost_map,
            "thr_npv": thr_npv,
            "eval_cost": eval_cost,
            "eval_npv": eval_npv,
            "fairness": fairness_tables
        }

    return results

In [ ]:
# === 6) Ejecuta por horizontes y resume ===
# Si no existe el contenedor global, créalo
if "ALL_RESULTS" not in globals():
    ALL_RESULTS = {}

# Definimos los 4 horizontes que estás usando
HORIZONS = [
    ("W0_discrete", "y_w0", 0, 1),   # primeras horas
    ("W1_discrete", "y_w1", 0, 1),   # resto del día 1
    ("W7_discrete", "y_w7", 1, 7),   # hasta 7 días
    ("W30_discrete", "y_w30", 7, 30) # hasta 30 días
]

for hname, lcol, gap_days, embargo_days in HORIZONS:
    if hname in ALL_RESULTS and "HistGB" in ALL_RESULTS[hname] and "model" in ALL_RESULTS[hname]["HistGB"]:
        continue

    # 1) entrena y evalúa con tu función del notebook
    res = train_eval_horizon(
        hname,
        lcol,
        gap_before_days=gap_days,
        embargo_after_days=embargo_days,
        cost_fp=2.0,
        cost_fn=2.0,
        calib_method="isotonic"
    )
    # 2) re-entrena un modelo "full data" para producción y guárdalo
    X, y, feats, num_cols, cat_cols, k = get_xy_for_horizon(hname, lcol)  # :contentReference[oaicite:2]{index=2}
    pre_dense = onehot_preprocessor(num_cols, cat_cols, scale_numeric=False, min_freq=0.01, dense_ohe=True)
    full_model = make_histgb(pre_dense)
    full_model.fit(X[feats], y)

    # 3) cuélgalo en el resultado
    if "HistGB" not in res:
        res["HistGB"] = {}
    res["HistGB"]["model"] = full_model

    ALL_RESULTS[hname] = res

    
#Helper para sacar las probas OOF/producción


def get_oof_scores(hname, label_col):
    # recupera exactamente el mismo X que usaste al entrenar
    X, y, feats, num_cols, cat_cols, k = get_xy_for_horizon(hname, label_col)
    # toma el modelo que acabamos de colgar
    model = ALL_RESULTS[hname]["HistGB"]["model"]
    proba = model.predict_proba(X[feats])[:, 1]
    return X.index, proba


# Scores por horizonte

idx_w0, p_w0   = get_oof_scores("W0_discrete",  "y_w0")
idx_w1, p_w1   = get_oof_scores("W1_discrete",  "y_w1")
idx_w7, p_w7   = get_oof_scores("W7_discrete",  "y_w7")
idx_w30, p_w30 = get_oof_scores("W30_discrete", "y_w30")

# base de usuarios: la misma que tus labels
df_scores = df_lab[["user_id", "signup_date", "channelDetail"]].copy()

df_scores.loc[idx_w0,  "p_w0"]  = p_w0
df_scores.loc[idx_w1,  "p_w1"]  = p_w1
df_scores.loc[idx_w7,  "p_w7"]  = p_w7
df_scores.loc[idx_w30, "p_w30"] = p_w30

#Derivar scores día / semana / mes
# día = ventanas en las primeras 24h (W0 + W1)
df_scores["score_dia"] = (
    df_scores["p_w0"].fillna(0.0) +
    df_scores["p_w1"].fillna(0.0)
).clip(0, 1)

# semana = lo anterior + lo que puede caer entre 24h y 7d
df_scores["score_semana"] = (
    df_scores["score_dia"] +
    df_scores["p_w7"].fillna(0.0)
).clip(0, 1)

# mes = todo lo de arriba + 7d..30d
df_scores["score_mes"] = (
    df_scores["score_semana"] +
    df_scores["p_w30"].fillna(0.0)
).clip(0, 1)

#Segmentación de probabilidad de conocer su primera transacción
def bucket_prob(p):
    if p >= 0.70:
        return " alta"
    elif p >= 0.40:
        return " media"
    else:
        return " baja"

df_scores["prob_first_tx_dia"]    = df_scores["score_dia"]
df_scores["prob_first_tx_semana"] = df_scores["score_semana"]
df_scores["prob_first_tx_mes"]    = df_scores["score_mes"]

df_scores["seg_first_tx_dia"]    = df_scores["prob_first_tx_dia"].apply(bucket_prob)
df_scores["seg_first_tx_semana"] = df_scores["prob_first_tx_semana"].apply(bucket_prob)
df_scores["seg_first_tx_mes"]    = df_scores["prob_first_tx_mes"].apply(bucket_prob)

# Canal de transacción más viable por ventana
# el dataset original sí traía info de la primera tx pero la sacaste de X para evitar leakage
TX_COL = "first_tx_type" if "first_tx_type" in df_lab.columns else (
    "activation_channel" if "activation_channel" in df_lab.columns else None
)

canal_dia = None
canal_semana = None
canal_mes = None

if TX_COL is not None:
    # quién SÍ activó en cada ventana
    m_dia = (df_lab["y_w0"] == 1) | (df_lab["y_w1"] == 1)
    m_sem = m_dia | (df_lab["y_w7"] == 1)
    m_mes = m_sem | (df_lab["y_w30"] == 1)

    # modo por canal de origen
    canal_dia = (
        df_lab.loc[m_dia]
        .groupby("channelDetail")[TX_COL]
        .agg(lambda s: s.value_counts().index[0] if s.notna().any() else None)
        .rename("canal_tx_mas_viable_dia")
    )

    canal_semana = (
        df_lab.loc[m_sem]
        .groupby("channelDetail")[TX_COL]
        .agg(lambda s: s.value_counts().index[0] if s.notna().any() else None)
        .rename("canal_tx_mas_viable_semana")
    )

    canal_mes = (
        df_lab.loc[m_mes]
        .groupby("channelDetail")[TX_COL]
        .agg(lambda s: s.value_counts().index[0] if s.notna().any() else None)
        .rename("canal_tx_mas_viable_mes")
    )

    df_scores = (
        df_scores
        .merge(canal_dia,    on="channelDetail", how="left")
        .merge(canal_semana, on="channelDetail", how="left")
        .merge(canal_mes,    on="channelDetail", how="left")
    )

# df_scores ahora trae TODO lo que pediste
df_scores.head()


def summarize_results(allres):
    rows = []

    for hname, d in allres.items():
        # 1) a veces guardaste un string tipo "Models fail"
        if not isinstance(d, dict):
            print(f"[WARN] {hname}: no es dict ({type(d)}). Lo salto.")
            continue

        for m, r in d.items():
            # 2) a veces el valor del modelo tampoco es dict
            if not isinstance(r, dict):
                print(f"[WARN] {hname}/{m}: valor no dict -> {r}. Lo salto.")
                continue

            # 3) checa que tenga las llaves mínimas
            oof_ap  = r.get("oof_ap")
            oof_auc = r.get("oof_auc")
            thr_cost = r.get("thr_cost")
            thr_npv  = r.get("thr_npv")

            eval_cost = r.get("eval_cost") or {}
            eval_npv  = r.get("eval_npv") or {}

            # si no tiene métrica base, no lo metas
            if oof_ap is None or oof_auc is None:
                print(f"[WARN] {hname}/{m}: sin métricas OOF. Lo salto.")
                continue

            rows.append({
                "horizon":  hname,
                "model":    m,
                "AUC-PR":   oof_ap,
                "AUC-ROC":  oof_auc,
                "thr_cost": thr_cost,
                "thr_npv":  thr_npv,
                "NPV@cost": eval_cost.get("npv"),
                "Prec@cost": eval_cost.get("precision"),
                "Rec@cost":  eval_cost.get("recall"),
                "NPV@npv":  eval_npv.get("npv"),
                "Prec@npv": eval_npv.get("precision"),
                "Rec@npv":  eval_npv.get("recall"),
            })

    if not rows:
        print("[WARN] summarize_results: no hay filas válidas.")
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values(["horizon", "AUC-PR"], ascending=[True, False])
        .reset_index(drop=True)
    )

summary_df = summarize_results(ALL_RESULTS)
display(summary_df)



=== Horizonte: W0_discrete | label=y_w0 ===

-- Modelo: HistGB --
   Fold 1: AUC-PR=0.3719 | AUC-ROC=0.6903
   Fold 2: AUC-PR=0.4195 | AUC-ROC=0.7079
   Fold 3: AUC-PR=0.4371 | AUC-ROC=0.6803
   Fold 4: AUC-PR=0.4325 | AUC-ROC=0.6775
[HistGB] OOF: AUC-PR=0.4188 | AUC-ROC=0.6916
